In [ ]:
from pathlib import Path
import os
PROJECT_ROOT = Path(os.environ["DIGIMAP_CODE_ROOT"]).resolve()
DATA_ROOT = Path(os.environ["DIGIMAP_DATA_ROOT"]).resolve()
RUN_ID = os.environ["DIGIMAP_RUN_ID"]
RUN_DIR = DATA_ROOT / "outputs" / RUN_ID

In [ ]:
import sys, csv, json, html
from IPython.display import display, Markdown, JSON, Image, HTML
sys.path.insert(0, str(PROJECT_ROOT))
from src.pipeline import verify_run
verification = verify_run(RUN_DIR, code_root=PROJECT_ROOT, data_root=DATA_ROOT, require_success=True)
assert verification["run_id"] == RUN_ID
REPORT = RUN_DIR / "report" / "main"
SUMMARY = RUN_DIR / "summarize" / "main"
display(JSON(verification))
display(Markdown((REPORT / "SUMMARY.md").read_text()))

In [ ]:
inspection = json.loads((RUN_DIR / "inspect" / "inputs" / "main" / "input_inspection.json").read_text())
display(JSON({"scope": inspection["scope"], "poi_records": inspection["poi"]["scanned_count"],
              "station_records": inspection["stations"]["scanned_count"],
              "poi_issues": inspection["poi"]["issue_counts"], "input_bytes_unchanged": inspection["input_bytes_unchanged"]}))
def read_csv(path):
    with path.open(newline="", encoding="utf-8") as stream:
        return list(csv.DictReader(stream))
def table(rows, limit=15):
    if not rows:
        display(Markdown("No rows in this run/scope.")); return
    names = list(rows[0])
    content = "<table><tr>" + "".join("<th>"+html.escape(k)+"</th>" for k in names)+"</tr>"
    for row in rows[:limit]:
        content += "<tr>"+"".join("<td>"+html.escape(str(row[k]))+"</td>" for k in names)+"</tr>"
    display(HTML(content+"</table>"))
table(read_csv(SUMMARY / "radius_summary.csv"))
table(read_csv(SUMMARY / "station_radius_totals.csv"))

In [ ]:
sample = read_csv(REPORT / "review" / "sample_stations.csv")
table(sample)
if sample:
    station_key = sample[0]["station_key"]
    display(Markdown("Selected station: `"+station_key+"`"))
    table([r for r in read_csv(REPORT / "review" / "sample_composition.csv") if r["station_key"] == station_key])
    table([r for r in read_csv(REPORT / "review" / "sample_matches.csv") if r["station_key"] == station_key])

In [ ]:
display(JSON(json.loads((REPORT / "validation" / "independent_validation.json").read_text()), expanded=False))
table(read_csv(REPORT / "validation" / "original_station_trace.csv"))
table(read_csv(REPORT / "validation" / "original_record_trace.csv"))
for image in ("composition_and_counts.png", "station_map.png"):
    display(Image(filename=str(REPORT / "figures" / image)))

In [ ]:
final_verification = verify_run(RUN_DIR, code_root=PROJECT_ROOT, data_root=DATA_ROOT, require_success=True)
display(JSON(final_verification))